# Визуализация результатов модели ConvBiGRUDetector

Этот ноутбук предназначен для визуализации результатов работы модели ConvBiGRUDetector и сравнения их с оригинальной разметкой.

## Импорт необходимых библиотек

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

# Добавляем путь к src для импорта модулей
PROJECT_ROOT = Path.cwd().resolve().parent  # если ноутбук лежит в notebooks
sys.path.insert(0, str(PROJECT_ROOT))

# Импорты из проекта
from src.modeling.model_registry import get_model
from src.data_loading.epilepsy_datamodule import EpilepsyDataModule
from src.data_loading.seizure_annotation_reader import SeizureAnnotationReader

# Импорт функций постобработки
from experiments.postprocess_predictions import postprocess_predictions, predict_full_recording

# Параметры отображения
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 12

## Загрузка модели

In [ ]:
#ConvBiGRUDetector - last
#MSConvBiGRUDetector - last-v1 
model = get_model('RDSCBiGRUDetector', {})
print(f"Модель загружена: {type(model).__name__}")
print(f"Количество параметров: {sum(p.numel() for p in model.parameters()):,}")

ckpt_path = "../experiments/exp_002/checkpoints/RDSCBiGRUDetector-best-val_loss=0.00101.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu")

# если это Lightning-чекпойнт, веса обычно тут:
state_dict = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt

# убираем префикс "model."
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("model."):
        new_state_dict[k[len("model."):]] = v
    else:
        new_state_dict[k] = v  # на случай, если часть ключей без префикса
model.load_state_dict(new_state_dict)
model.eval()

## Загрузка данных

In [ ]:
# Инициализация датамодуля

train_animals=["Ati5x1", "Dex1x2NE", "Ati5y2"]  # Список ID животных для обучения
val_animals=["Ati5y1"]    # Список ID животных для валидации
test_animals=["Dex4x5"]   # Список ID животных для тестирования

data_module = EpilepsyDataModule(
    data_dir="../data/processed",
    window_length=2000,
    batch_size=64,
    overlap=0,
     train_animals=train_animals,
     val_animals=val_animals,
     test_animals=test_animals,
)

# Подготовка данных
data_module.prepare_data()
data_module.setup('test')

print(f"Размер тестовой выборки: {len(data_module.test_dataset)}")

## Визуализация результатов

In [ ]:
def probs_to_segments(predicted_probs, threshold=0.5, min_len=1):
    """
    Превращает массив вероятностей в список сегментов (start, end) по условию prob >= threshold.
    start/end — индексы сэмплов, end не включительно (как в Python slicing).
    min_len — минимальная длина сегмента в сэмплах (чтобы отфильтровать короткий шум).
    """
    mask = np.asarray(predicted_probs) >= threshold
    if mask.size == 0:
        return []

    # Находим границы участков True
    diff = np.diff(mask.astype(np.int8))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1

    # Если начинается с True — старт с 0
    if mask[0]:
        starts = np.r_[0, starts]
    # Если заканчивается True — конец = len(mask)
    if mask[-1]:
        ends = np.r_[ends, mask.size]

    segments = [(int(s), int(e)) for s, e in zip(starts, ends) if (e - s) >= min_len]
    return segments


def visualize_prediction(signal, true_labels, predicted_probs, segments=None,
                        threshold=0.5, fs=400, title="Результаты предсказания",
                        min_seg_len=1):
    """Визуализация сигнала, истинных меток, вероятностей и предсказанных сегментов по threshold."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

    # Время
    n = len(signal[0])
    time = np.arange(n) / fs

    # Сигнал (1-й канал)
    ax1.plot(time, signal[0], 'b-', linewidth=0.8, label='Сигнал (канал 1)')
    ax1.set_ylabel('Амплитуда')
    ax1.set_title(f'{title} - Сигнал')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Истинные метки + вероятности
    ax2.plot(time, true_labels, 'g-', linewidth=1, label='Истинные метки')
    ax2.plot(time, predicted_probs, 'r-', linewidth=1, label='Предсказанные вероятности')

    # Линия threshold
    ax2.axhline(threshold, color='k', linestyle='--', linewidth=1, alpha=0.7,
                label=f'Threshold = {threshold:.2f}')

    # Сегменты: либо используем переданные, либо считаем по threshold
    if segments is None:
        segments = probs_to_segments(predicted_probs, threshold=threshold, min_len=min_seg_len)

    # Отрисовка сегментов
    first = True
    for start, end in segments:
        start_time = start / fs
        end_time = end / fs
        ax2.axvspan(
            start_time, end_time,
            color='yellow', alpha=0.3,
            label='Предсказанные сегменты' if first else None
        )
        first = False

    ax2.set_xlabel('Время (секунды)')
    ax2.set_ylabel('Вероятность / Метки')
    ax2.set_title(f'{title} - Предсказания и истинные метки')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Пример предсказания на тестовых данных

In [ ]:
# Получение случайного примера из тестовой выборки
test_sample = data_module.test_dataset[347]
signal, true_label = test_sample

# Добавляем размерность батча
signal_batch = signal.unsqueeze(0)

# Получение предсказания
model.eval()
with torch.no_grad():
    logits = model(signal_batch)
    predicted_probs = torch.sigmoid(logits).squeeze().numpy()

# Постобработка
segments = postprocess_predictions(predicted_probs)

# Визуализация
visualize_prediction(
    signal.numpy(), 
    true_label.numpy(), 
    predicted_probs, 
    None, 
    title="Пример предсказания ConvBiGRUDetector"
)